In [25]:
import os
import s3fs
import pandas as pd

os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"] = "when_required"
os.environ["AWS_RESPONSE_CHECKSUM_VALIDATION"] = "when_required"

fs = s3fs.S3FileSystem(
    client_kwargs={"endpoint_url": "https://" + os.environ["AWS_S3_ENDPOINT"]},
    skip_instance_cache=True,
)

BUCKET = next(
    b for b in ["samuelbarre", "acharki-ensae"]
    if fs.exists(f"{b}/Projet-analyse-BPE/BPE25.parquet")
)

In [26]:
chemin_bpe = f"{BUCKET}/Projet-analyse-BPE/processed/bpe_alim_2025.parquet"
chemin_dvf = f"{BUCKET}/Projet-analyse-BPE/processed/dvf_2025.parquet"

with fs.open(chemin_bpe, "rb") as f:
    bpe = pd.read_parquet(f)

with fs.open(chemin_dvf, "rb") as f:
    dvf = pd.read_parquet(f)

print("BPE :", bpe.shape)
print("DVF :", dvf.shape)

BPE : (135587, 21)
DVF : (930120, 14)


In [27]:
chemin_insee = f"{BUCKET}/Projet-analyse-BPE/fichier_diffusion_2025.xlsx"

with fs.open(chemin_insee, "rb") as f:
    urbanisation = pd.read_excel(
        f,
        sheet_name="Maille communale",
        header=4,
        dtype={"CODGEO": str}
    )

urbanisation.head()

,CODGEO,LIBGEO,DENS,LIBDENS,PMUN22,P1,P2,P3,DENS_AAV,LIBDENS_AAV,DENS7,LIBDENS7
0,01001,L'Abergement-Clémenciat,3,Rural,859,0.0,0.000000,100.000000,4,Rural non périurbain,6,Rural à habitat dispersé
1,01002,L'Abergement-de-Varey,3,Rural,273,0.0,0.000000,100.000000,4,Rural non périurbain,6,Rural à habitat dispersé
2,01004,Ambérieu-en-Bugey,2,Urbain intermédiaire,15554,0.0,96.933762,3.066238,2,Urbain intermédiaire,2,Centres urbains intermédiaires
3,01005,Ambérieux-en-Dombes,3,Rural,1917,0.0,0.000000,100.000000,3,Rural périurbain,5,Bourgs ruraux
4,01006,Ambléon,3,Rural,114,0.0,0.000000,100.000000,4,Rural non périurbain,6,Rural à habitat dispersé


In [28]:
urbanisation = urbanisation[["CODGEO", "DENS", "LIBDENS"]]

urbanisation.head()

,CODGEO,DENS,LIBDENS
0,01001,3,Rural
1,01002,3,Rural
2,01004,2,Urbain intermédiaire
3,01005,3,Rural
4,01006,3,Rural


In [29]:
dvf["code_insee"] = dvf["code_commune"].copy()

# Regrouper les arrondissements dans leur commune
dvf.loc[dvf["code_insee"].between("75101", "75120"), "code_insee"] = "75056"
dvf.loc[dvf["code_insee"].between("69381", "69389"), "code_insee"] = "69123"
dvf.loc[dvf["code_insee"].between("13201", "13216"), "code_insee"] = "13055"

In [30]:
dvf = dvf.merge(
    urbanisation,
    left_on="code_insee",
    right_on="CODGEO",
    how="left",
    validate="many_to_one"
).drop(columns="CODGEO")

In [31]:
dvf.head(10)

,id_mutation,date_mutation,nature_mutation,valeur_fonciere,code_commune,nom_commune,code_departement,type_local,surface_reelle_bati,surface_terrain,longitude,latitude,prix_m2,geometry,code_insee,DENS,LIBDENS
0,2025-1,2025-01-07,Vente,468000.0,01158,Farges,01,Maison,111,133.0,5.907199,46.170775,4216.216216,"b'\x01\x01\x00\x00\x00Hi\xeb\xaa\xaf4,Amz\xa7\...",01158,3,Rural
1,2025-2,2025-01-06,Vente,180000.0,01257,Montanges,01,Appartement,87,76.0,5.801490,46.164545,2068.965517,b'\x01\x01\x00\x00\x00\x1a\xd3\xb5\xd33\xf5+A\...,01257,3,Rural
2,2025-3,2025-01-06,Vente,490000.0,01065,Buellas,01,Maison,190,500.0,5.155043,46.206188,2578.947368,b'\x01\x01\x00\x00\x00\xc0\xe4\xc1%\xb7n*A[*i\...,01065,3,Rural
3,2025-4,2025-01-09,Vente,183000.0,01173,Gex,01,Appartement,35,NaN,6.064268,46.332953,5228.571429,"b'\x01\x01\x00\x00\x00\x91\x84\x0fo\xd5\x8d,A\...",01173,2,Urbain intermédiaire
4,2025-5,2025-01-07,Vente,295000.0,01028,Baneins,01,Maison,95,965.0,4.910721,46.121708,3105.263158,b'\x01\x01\x00\x00\x00\xf4\xac\x08\xb7]\xdd)A\...,01028,3,Rural
5,2025-7,2025-01-08,Vente,131937.3,01005,Amberieux-en-Dombes,01,Appartement,49,NaN,4.905917,45.995081,2692.597959,b'\x01\x01\x00\x00\x00\x8dw\xf7\xc9\x1e\xdd)A]...,01005,3,Rural
6,2025-8,2025-01-09,Vente,173000.0,01306,Pont-de-Veyle,01,Appartement,78,NaN,4.887005,46.260695,2217.948718,b'\x01\x01\x00\x00\x00E\x11Mh.\xcc)A\xd9<\r>\x...,01306,3,Rural
7,2025-9,2025-01-06,Vente,425000.0,01318,Rance,01,Maison,110,500.0,4.870114,45.973288,3863.636364,b'\x01\x01\x00\x00\x00\xfa\x17r\xab\xec\xc7)A\...,01318,3,Rural
8,2025-10,2025-01-09,Vente,1165000.0,01419,Thoiry,01,Maison,190,860.0,5.981172,46.243363,6131.578947,"b'\x01\x01\x00\x00\x00\xf6\xc1|\x96\xdd^,A\xdf...",01419,2,Urbain intermédiaire
9,2025-11,2025-01-07,Vente,630000.0,01308,Pougny,01,Maison,100,761.0,5.950677,46.136450,6300.000000,"b'\x01\x01\x00\x00\x00y\\\xcc\xa5\xfcO,As\xa7\...",01308,3,Rural


In [32]:
print(dvf["LIBDENS"].value_counts(dropna=False))

LIBDENS
Urbain dense            340141
Rural                   311352
Urbain intermédiaire    278627
Name: count, dtype: int64


In [ ]:
import numpy as np
from sklearn.neighbors import BallTree

# Rayon de la Terre en m
R = 6371000

coord_dvf = np.radians(dvf[["latitude", "longitude"]].to_numpy())
coord_bpe = np.radians(bpe[["LATITUDE", "LONGITUDE"]].to_numpy())

# Construire l'arbre des commerces
arbre = BallTree(coord_bpe, metric="haversine")

dvf["commerces_500m"] = arbre.query_radius(
    coord_dvf,
    r=500 / R,
    count_only=True
)

distances, indices = arbre.query(coord_dvf, k=1)
dvf["distance_commerce_m"] = distances[:, 0] * R

dvf[["commerces_500m", "distance_commerce_m"]].head(10)

,commerces_500m,distance_commerce_m
0,0,870.463654
1,0,2303.620286
2,0,1093.225261
3,3,274.800147
4,0,1935.882139
5,3,74.260125
6,4,42.104244
7,0,3547.864870
8,1,296.631891
9,0,2951.043402


In [37]:
dvf[["commerces_500m", "distance_commerce_m"]].describe()

,commerces_500m,distance_commerce_m
count,930120.000000,9.301200e+05
mean,12.714714,6.255749e+02
std,26.384550,3.884879e+03
min,0.000000,3.640851e-02
25%,0.000000,9.969786e+01
50%,3.000000,2.371424e+02
75%,11.000000,5.678347e+02
max,283.000000,3.606995e+06


In [38]:
dvf.nlargest(10, "distance_commerce_m")[
    ["nom_commune", "code_departement",
     "distance_commerce_m", "commerces_500m"]
]

,nom_commune,code_departement,distance_commerce_m,commerces_500m
856516,Issy-les-Moulineaux,92,3.606995e+06,0
856872,Vaucresson,92,1.691870e+05,0
855956,Vaucresson,92,1.690181e+05,0
371022,Bourriot-Bergonce,40,1.441620e+04,0
370042,Bourriot-Bergonce,40,1.436034e+04,0
374375,Bourriot-Bergonce,40,1.435374e+04,0
369590,Bourriot-Bergonce,40,1.426223e+04,0
459039,Saint-Ouen-Domprot,51,1.341388e+04,0
459272,Saint-Ouen-Domprot,51,1.324008e+04,0
583023,Borce,64,1.289183e+04,0


In [39]:
dvf.nlargest(10, "distance_commerce_m")[
    ["nom_commune", "longitude", "latitude",
     "distance_commerce_m"]
]

,nom_commune,longitude,latitude,distance_commerce_m
856516,Issy-les-Moulineaux,0.269463,83.517658,3.606995e+06
856872,Vaucresson,2.287487,40.882084,1.691870e+05
855956,Vaucresson,2.285057,40.883919,1.690181e+05
371022,Bourriot-Bergonce,-0.186952,44.121401,1.441620e+04
370042,Bourriot-Bergonce,-0.160891,44.116882,1.436034e+04
374375,Bourriot-Bergonce,-0.191450,44.124067,1.435374e+04
369590,Bourriot-Bergonce,-0.198201,44.152456,1.426223e+04
459039,Saint-Ouen-Domprot,4.412537,48.609575,1.341388e+04
459272,Saint-Ouen-Domprot,4.407830,48.610175,1.324008e+04
583023,Borce,-0.557828,42.857609,1.289183e+04


Erreur :
Issy-les-Moulineaux latitude = 48°49′26″N
Vaucresson latitude = 48.83

On retire les 3 valeurs absurdes (856516, 856872 & 855956)

In [43]:
dvf_reg = dvf.drop(index=[856516, 856872, 855956]).copy()


In [45]:
import statsmodels.formula.api as smf

dvf_reg["log_prix_m2"] = np.log(dvf_reg["prix_m2"])

modele1 = smf.ols(
    "log_prix_m2 ~ commerces_500m + surface_reelle_bati + C(type_local) + C(code_departement)",
    data=dvf_reg
).fit(cov_type="HC3")

print(modele1.summary())

                            OLS Regression Results                            
Dep. Variable:            log_prix_m2   R-squared:                       0.402
Model:                            OLS   Adj. R-squared:                  0.402
Method:                 Least Squares   F-statistic:                     7426.
Date:                Tue, 10 Nov 2026   Prob (F-statistic):               0.00
Time:                        22:09:33   Log-Likelihood:            -8.3127e+05
No. Observations:              930117   AIC:                         1.663e+06
Df Residuals:                  930021   BIC:                         1.664e+06
Df Model:                          95                                         
Covariance Type:                  HC3                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             